In [1]:
from Environment import *
from DDPG import *
from NN_Module import *
from config import Config
import os

import numpy as np
from numpy import linalg as LA
from tqdm import tqdm
import torch
import matplotlib.pyplot as plt

from loguru import logger
from scipy.io import loadmat

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
print(f"Using device: {device}")

from loguru import logger

### a simple logger
logger.remove()
logger.add(sys.stderr, level='INFO')

Using device: cuda


4

In [2]:
env_seed = 5        #10-h  5-h 0-l 1-h 2-l 3-l 4l 7h 8h 9l
episode_num = 3000   # the total test episode
step_num = 200      # the longest test step

### create testing environment
injection_bus = np.array([9, 10, 15, 19, 32, 35, 47, 58, 65, 74, 82, 91, 103, 60]) #11, 36, 75,/ 1,5,9
pp_net = create_123bus()
env = Env_123bus(pp_net, injection_bus)
state, topology, senario = env.reset_topo(seed=env_seed)
topology = torch.tensor(topology, dtype=torch.float32, device='cuda').unsqueeze(0)
agent_num = env.agentnum
# pf_res_plotly(pp_net);

d:\Code\Python\Flexible_Voltage_Control\.venv\Lib\site-packages\pandapower\converter\pypower\from_ppc.py:334: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  branch_lookup.loc[is_trafo, "element"] = idx_trafo


### Some Plot Function

In [3]:
# plot policy
def plot_policy(policy_net, topology):
    fig, axs = plt.subplots(2, 7, figsize=(15,6))
    title = ['Bus 9', 'Bus 10', 'Bus 15', 'Bus19', 'Bus 32', 'Bus 35', 'Bus 47', 
                'Bus 58', 'Bus 65', 'Bus 74', 'Bus 72', 'Bus 91', 'Bus 103', 'Bus 60']
    for i in range(agent_num):
        axs[i//7][i%7].clear()
        # plot policy
        N = 40
        s_array = np.zeros(N,)
        
        a_array_baseline = np.zeros(N,)
        a_array = np.zeros(N,)
        
        for j in range(N):
            state = torch.tensor([[0.80+0.01*j]])
            s_array[j] = state

            action_baseline = (np.maximum(state.cpu()-1.05, 0)-np.maximum(0.95-state.cpu(), 0)).reshape((1,))
        
            action = policy_net[i](state, topology)
            action = action.detach().cpu().numpy()[0]
            
            a_array_baseline[j] = -action_baseline[0]
            a_array[j] = -action

        axs[i//7][i%7].plot(12*s_array, 50*a_array_baseline, '-.', label = 'Linear')
        axs[i//7][i%7].plot(12*s_array, a_array, label = 'Flexible-DDPG')
        axs[i//7][i%7].set_title(title[i])
        axs[i//7][i%7].legend(loc='lower left')

def plot_safe_net(net):
    fig, axs = plt.subplots(2, 7, figsize=(15,6))
    title = ['Bus 9', 'Bus 10', 'Bus 15', 'Bus19', 'Bus 32', 'Bus 35', 'Bus 47', 
                'Bus 58', 'Bus 65', 'Bus 74', 'Bus 72', 'Bus 91', 'Bus 103', 'Bus 60']
    for i in range(agent_num):
        N = 40
        s_array = np.zeros(N,)
        
        a_array_baseline = np.zeros(N,)
        a_array = np.zeros(N,)
        
        for j in range(N):
            state = np.array([0.8+0.01*j])
            s_array[j] = state

            action_baseline = (np.maximum(state-1.05, 0)-np.maximum(0.95-state, 0)).reshape((1,))
        
            action = net[i].get_action([state])
            
            a_array_baseline[j] = -action_baseline[0]
            a_array[j] = -action

        axs[i//7][i%7].plot(12*s_array, 2*a_array_baseline, '-.', label = 'Linear')
        axs[i//7][i%7].plot(12*s_array, a_array, label = 'Stable-DDPG')
        axs[i//7][i%7].legend(loc='lower left')

def plot_x_policy(policy_net):
    """
    用 Plotly 绘制不同拓扑下的 RLC-FT 策略曲线，所有情形绘制在单个图中
    """
    import plotly.graph_objects as go
    fig = go.Figure()
    N = 400
    for i in range(5):
        policy_vals = []
        # 对于每个拓扑情形，通过 reset_topo 获得对应拓扑设定
        state, topo, senario = env.reset_topo(seed=i)
        topo_tensor = torch.cuda.FloatTensor(topo).unsqueeze(0)
        for j in range(N):
            state_tensor = torch.tensor([[0.80 + 0.001 * j]])
            action_tensor = policy_net[2](state_tensor, topo_tensor).detach()
            policy_vals.append(float(-action_tensor.view(-1)[0].cpu()))

        x_vals = np.linspace(10, 14, N)
        fig.add_trace(go.Scatter(x=x_vals, y=policy_vals,
                                 mode='lines',
                                 name=f'Topology {i}'))
    fig.update_layout(
        font=dict(size=16),
        width=700,
        height=500,
        # margin=dict(l=30, r=30, t=30, b=30),   # Uncomment to adjust margins
        margin=dict(r=30,t=30,b=60),
        xaxis_title='Voltage (kV)',
        yaxis_title='Q (MVar)',
        xaxis=dict(
            showgrid=True,
            tickfont=dict(size=12),
        ),
        yaxis=dict(
            tickfont=dict(size=12),
            showgrid=True,
            zeroline=True,
            zerolinecolor='lightgray'
        ),
        legend=dict(
            x=1,
            y=1,
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(255,255,255,1.0)'
        ),
    )

    fig.show()

### Load model

In [4]:

agent_policy_net = []
safe_agent_net = []

### load nn model parameter from saved model 
for i in range(agent_num):
    topology_net = TopologyNet(topology_dim=env.topology_dim, output_dim=1, hidden_dim=Config.topology_hidden_dim)
    policy_net = FlexiblePolicyNet(env=env, topology_net=topology_net, obs_dim=1, action_dim=1, hidden_dim=Config.hidden_dim_123bus).to(device)
    agent_policy_net.append(policy_net)

for i in range(agent_num):
    policy_net = SafePolicyNetwork(env=env, obs_dim=1, action_dim=1, hidden_dim=100).to(device)
    safe_agent_net.append(policy_net)

for i in range(agent_num):
    #value_net_dict = torch.load(f'check_points/value_net/2023-06-19/Step_200_Seed_12_a{i}.pth')
    # policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2023-09-12/Step_800_Seed_18_a{i}.pth'))
    policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-08-19/Step_950_Seed_2_a{i}.pth'))

    agent_policy_net[i].load_state_dict(policy_net_dict)

for i in range(agent_num):
    #value_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/2023-06-19/SafeDDPG_value_Step_200_a{i}.pth')
    policy_net_dict = torch.load(f'D:/Code/Python/Stable-DDPG-for-voltage-control/checkpoints/single-phase/123bus/safe-ddpg/policy_net_checkpoint_a{i}.pth')
    

    safe_agent_net[i].load_state_dict(policy_net_dict)

In [5]:
# plot_policy(agent_policy_net, topology)
plot_x_policy(agent_policy_net)
# plot_safe_net(safe_agent_net)

C:\Users\wdyao\AppData\Local\Temp\ipykernel_10476\3240596122.py:69: UserWarning:

The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\tensor\python_tensor.cpp:80.)



### Flexible NN Contoller

In [6]:
### test our controller
voltage = []
q = []
cost = []
success_list = []
fail_list = []
entire_list = []
control_cost = []
reward_list = []
object_cost = []
voltage_trajs = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    abnormal_stop = False
    voltage_episode = []   # stores full voltage vectors for this episode

    for step in range(step_num):
        action = []
        for i in range(agent_num):
            action_agent = 0.5 * agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
            action_agent = action_agent.detach().cpu().numpy()[0]
            action.append(action_agent)

        action = last_action - np.asarray(action)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            success_list.append((episode,step))
            logger.success('episode {} stable at {} steps',success_list[-1][0], success_list[-1][1])
            break

        voltage.append(state)
        voltage_episode.append(state.copy())  # record full state vector

        q.append(action)

        state = next_state
        
        episode_reward += reward
        
        cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    reward_list.append(episode_reward)
    control_cost.append(episode_control)
    object_cost.append(np.sum(cost))
    if len(voltage_episode) > 0 and senario == 0:
        voltage_trajs.append(np.vstack(voltage_episode))

    if (not done) and (abnormal_stop == False):
        entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(success_list))
logger.info('total fail episode is {}', len(fail_list))
logger.info('number of finished at entire episode is {}', len(entire_list))

2026-03-18 15:57:34.752 | SUCCESS  | __main__:<module>:49 - episode 0 stable at 35 steps
2026-03-18 15:57:35.198 | SUCCESS  | __main__:<module>:49 - episode 1 stable at 4 steps
2026-03-18 15:57:39.455 | SUCCESS  | __main__:<module>:49 - episode 2 stable at 60 steps
2026-03-18 15:57:43.246 | SUCCESS  | __main__:<module>:49 - episode 3 stable at 49 steps
2026-03-18 15:57:44.573 | SUCCESS  | __main__:<module>:49 - episode 4 stable at 16 steps
2026-03-18 15:57:47.060 | SUCCESS  | __main__:<module>:49 - episode 5 stable at 33 steps
2026-03-18 15:57:48.068 | SUCCESS  | __main__:<module>:49 - episode 6 stable at 12 steps
2026-03-18 15:57:50.253 | SUCCESS  | __main__:<module>:49 - episode 7 stable at 28 steps
2026-03-18 15:57:51.999 | SUCCESS  | __main__:<module>:49 - episode 8 stable at 22 steps
2026-03-18 15:57:52.711 | SUCCESS  | __main__:<module>:49 - episode 9 stable at 8 steps
2026-03-18 15:57:53.665 | SUCCESS  | __main__:<module>:49 - episode 10 stable at 11 steps
2026-03-18 15:57:55.25

In [21]:
success_list = np.array(success_list)
print('average recovery step is:')
print(np.mean(success_list[:,1]))
print(np.std(success_list[:,1]))
print('average reactive power cost is:')
print(np.mean(control_cost))
print(np.std(control_cost))
print('the total cost is:')
print(object_cost)
print(np.mean(object_cost))
print(np.std(object_cost))


average recovery step is:
17.993333333333332
12.09875843033307
average reactive power cost is:
271.48978385421833
287.7344639041968
the total cost is:
[np.float64(638.0469612694372), np.float64(704.3500501515182), np.float64(1832.1022066183857), np.float64(2668.723097933992), np.float64(2951.8126455271395), np.float64(3664.6519066906453), np.float64(3882.536728165563), np.float64(4401.5272174683305), np.float64(4941.052152600829), np.float64(5090.212942767468), np.float64(5331.806839081513), np.float64(5840.5603385197155), np.float64(6111.336683338739), np.float64(6770.251846138333), np.float64(6839.618244982174), np.float64(7097.378913470507), np.float64(7355.8770475470565), np.float64(8146.537782708579), np.float64(8419.818029159902), np.float64(8670.72386665051), np.float64(8838.293056695227), np.float64(8897.213548471045), np.float64(9091.804697426647), np.float64(9158.479396639363), np.float64(9461.051281354721), np.float64(9556.296753703406), np.float64(9649.211118376981), np.flo

In [10]:
### test our controller without topology change
voltage_ = []
q_ = []
cost_ = []
success_list_ = []
fail_list_ = []
entire_list_ = []
control_cost_ = []
reward_list_ = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = 1/env.topology_init
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    abnormal_stop = False

    for step in range(step_num):
        action = []
        for i in range(agent_num):
            action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
            action_agent = action_agent.detach().cpu().numpy()[0]
            action.append(action_agent)

        action = last_action - np.asarray(action)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            fail_list_.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage_ violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            fail_list_.append((episode,step))
            abnormal_stop = True
            break
        if done:
            success_list_.append((episode,step))
            logger.success('stable at {}',success_list_[-1])
            break

        voltage_.append(state)

        q_.append(action)

        state = next_state
        
        episode_reward += reward
        
        cost_.append(-reward)
        
        episode_control += LA.norm(action, 2)

    reward_list_.append(episode_reward)
    control_cost_.append(episode_control)

    if (not done) and (abnormal_stop == False):
        entire_list_.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(success_list_))
logger.info('total fail episode is {}', len(fail_list_))
logger.info('number of finished at entire episode is {}', len(entire_list_))

2025-08-19 16:15:30.671 | SUCCESS  | __main__:<module>:47 - stable at (0, 15)
2025-08-19 16:15:30.762 | SUCCESS  | __main__:<module>:47 - stable at (1, 0)
2025-08-19 16:15:32.503 | SUCCESS  | __main__:<module>:47 - stable at (2, 26)
2025-08-19 16:15:33.851 | SUCCESS  | __main__:<module>:47 - stable at (3, 21)
2025-08-19 16:15:34.396 | SUCCESS  | __main__:<module>:47 - stable at (4, 7)
2025-08-19 16:15:35.318 | SUCCESS  | __main__:<module>:47 - stable at (5, 14)
2025-08-19 16:15:35.662 | SUCCESS  | __main__:<module>:47 - stable at (6, 4)
2025-08-19 16:15:36.472 | SUCCESS  | __main__:<module>:47 - stable at (7, 12)
2025-08-19 16:15:37.117 | SUCCESS  | __main__:<module>:47 - stable at (8, 9)
2025-08-19 16:15:37.400 | SUCCESS  | __main__:<module>:47 - stable at (9, 3)
2025-08-19 16:15:37.724 | SUCCESS  | __main__:<module>:47 - stable at (10, 4)
2025-08-19 16:15:38.311 | SUCCESS  | __main__:<module>:47 - stable at (11, 8)
2025-08-19 16:15:38.707 | SUCCESS  | __main__:<module>:47 - stable at

In [11]:
success_list_ = np.array(success_list_)
print('average recovery step is:')
print(np.mean(success_list_[:,1]))
print(np.std(success_list_[:,1]))
print('average reactive power cost is:')
print(np.mean(control_cost_))
print(np.std(control_cost_))

average recovery step is:
7.27
5.5302591862105945
average reactive power cost is:
117.19120010127976
127.83208354634905


### baseline

In [12]:
### test the base line controller
voltage = []
q = []
cost = []
base_succ_list = []
base_fail_list = []
base_entire_list = []
base_control_cost = []
base_reward_list = []
base_object_cost = []
base_voltage_trajs = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    abnormal_stop = False

    for step in range(step_num):
        state1 = np.asarray(state-env.vmax)
        state2 = np.asarray(env.vmin-state)
        d_v = (np.maximum(state1, 0)-np.maximum(state2, 0)).reshape((agent_num,1))
        
        action = (last_action - 30*d_v)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            base_fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            base_fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            base_succ_list.append((episode,step))
            logger.success('stable at {}',base_succ_list[-1])
            break

        voltage.append(state)
        base_voltage_trajs.append(state.copy())  # record full state vector

        q.append(action)

        state = next_state
        
        episode_reward += reward
        
        cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    base_control_cost.append(episode_control)
    base_reward_list.append(episode_reward)
    base_object_cost.append(np.sum(cost))
    
    if (not done) and (abnormal_stop == False):
        base_entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(base_succ_list))
logger.info('total fail episode is {}', len(base_fail_list))
logger.info('number of finished at entire episode is {}', len(base_entire_list))

2025-08-19 16:44:45.383 | SUCCESS  | __main__:<module>:46 - stable at (0, 130)
2025-08-19 16:44:45.903 | SUCCESS  | __main__:<module>:46 - stable at (1, 41)
2025-08-19 16:44:48.211 | INFO     | __main__:<module>:68 - Episode 2 finish with entire step!
2025-08-19 16:44:50.525 | INFO     | __main__:<module>:68 - Episode 3 finish with entire step!
2025-08-19 16:44:51.615 | SUCCESS  | __main__:<module>:46 - stable at (4, 93)
2025-08-19 16:44:53.949 | INFO     | __main__:<module>:68 - Episode 5 finish with entire step!
2025-08-19 16:44:54.781 | SUCCESS  | __main__:<module>:46 - stable at (6, 67)
2025-08-19 16:44:56.955 | SUCCESS  | __main__:<module>:46 - stable at (7, 187)
2025-08-19 16:44:58.295 | SUCCESS  | __main__:<module>:46 - stable at (8, 114)
2025-08-19 16:44:59.625 | SUCCESS  | __main__:<module>:46 - stable at (9, 111)
2025-08-19 16:45:01.388 | SUCCESS  | __main__:<module>:46 - stable at (10, 152)
2025-08-19 16:45:03.311 | SUCCESS  | __main__:<module>:46 - stable at (11, 154)
2025-

In [16]:
base_succ_list = np.array(base_succ_list)
print('average recovery step is:')
print(np.mean(base_succ_list[:,1]))
print(np.std(base_succ_list[:,1]))
print('average reactive power cost is:')
print(np.mean(base_control_cost))
print(np.std(base_control_cost))
print('the total cost is:')
print(np.mean(base_object_cost))
print(np.std(base_object_cost))


average recovery step is:
106.380073800738
38.84028477016825
average reactive power cost is:
2034.3502486446
1696.9251531665395
the total cost is:
320150.5445877583
183398.67337175098


### Safe DDPG

In [14]:
### test the safe policy net
safe_voltage = []
safe_q = []
safe_cost = []
safe_succ_list = []
safe_fail_list = []
safe_entire_list = []
safe_contorl_cost = []
safe_reward_list = []
safe_object_cost = []
safe_voltage_trajs = []


for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    abnormal_stop = False

    for step in range(step_num):
        action = []
        for i in range(agent_num):
            action_agent = safe_agent_net[i].get_action(torch.cuda.FloatTensor([state[i]]).float().reshape(1,1))
            action.append(action_agent)
        
        action = last_action - 10*np.asarray(action).reshape((agent_num, 1))
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            safe_fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            safe_fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            safe_succ_list.append((episode,step))
            logger.success('stable at {}',safe_succ_list[-1])
            break
        safe_voltage.append(state)
        safe_voltage_trajs.append(state.copy())  # record full state vector

        safe_q.append(action)

        state = next_state
        
        episode_reward += reward
        
        safe_cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    safe_contorl_cost.append(episode_control)
    safe_reward_list.append(episode_reward)
    safe_object_cost.append(np.sum(safe_cost))
    if len(safe_voltage) > 0 and senario == 0:
        safe_voltage_trajs.append(np.vstack(safe_voltage))

    if (not done) and (abnormal_stop == False):
        safe_entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(safe_succ_list))
logger.info('total fail episode is {}', len(safe_fail_list))
logger.info('number of finished at entire episode is {}', len(safe_entire_list))


2025-08-19 16:52:38.336 | SUCCESS  | __main__:<module>:48 - stable at (0, 57)
2025-08-19 16:52:38.981 | SUCCESS  | __main__:<module>:48 - stable at (1, 16)
2025-08-19 16:52:42.485 | SUCCESS  | __main__:<module>:48 - stable at (2, 99)
2025-08-19 16:52:46.716 | SUCCESS  | __main__:<module>:48 - stable at (3, 118)
2025-08-19 16:52:48.134 | SUCCESS  | __main__:<module>:48 - stable at (4, 39)
2025-08-19 16:52:52.088 | SUCCESS  | __main__:<module>:48 - stable at (5, 107)
2025-08-19 16:52:53.067 | SUCCESS  | __main__:<module>:48 - stable at (6, 26)
2025-08-19 16:52:55.949 | SUCCESS  | __main__:<module>:48 - stable at (7, 80)
2025-08-19 16:52:57.633 | SUCCESS  | __main__:<module>:48 - stable at (8, 47)
2025-08-19 16:53:04.664 | INFO     | __main__:<module>:71 - Episode 9 finish with entire step!
2025-08-19 16:53:11.729 | INFO     | __main__:<module>:71 - Episode 10 finish with entire step!
2025-08-19 16:53:14.878 | SUCCESS  | __main__:<module>:48 - stable at (11, 88)
2025-08-19 16:53:21.940 | 

In [15]:
safe_succ_list = np.array(safe_succ_list)
print('average recovery step is:')
print(np.mean(safe_succ_list[:,1]))
print(np.std(safe_succ_list[:,1]))
print('average reactive power cost is:')
print(np.mean(safe_contorl_cost))
print(np.std(safe_contorl_cost))
print('the total cost is:')
print(np.mean(safe_object_cost))
print(np.std(safe_object_cost))

average recovery step is:
51.59124087591241
23.617868635805884
average reactive power cost is:
1087.4123612186024
1017.1948040388097
the total cost is:
177025.42608224635
101214.86131576673


In [25]:
import plotly.graph_objects as go
import numpy as np
from plotly.subplots import make_subplots

# Calculate statistics from raw data
methods = ['Linear', 'Safe-DDPG', 'RLC-FT']
metrics = ['Voltage Recovery Time', 'Reactive Power', 'Total Objective Cost']

# Extract data and calculate statistical values
data = {
    'Voltage Recovery Time': {
        'means': [
            np.mean(base_succ_list[:,1]),      # Linear
            np.mean(safe_succ_list[:,1]),      # Safe-DDPG
            np.mean(success_list[:,1])         # RLC-FT
        ],
        'stds': [
            np.std(base_succ_list[:,1]),       # Linear
            np.std(safe_succ_list[:,1]),       # Safe-DDPG
            np.std(success_list[:,1])          # RLC-FT
        ],
        'unit': 'Steps'
    },
    'Reactive Power': {
        'means': [
            np.mean(base_control_cost),        # Linear
            np.mean(safe_contorl_cost),        # Safe-DDPG (note original variable name spelling)
            np.mean(control_cost)              # RLC-FT
        ],
        'stds': [
            np.std(base_control_cost),         # Linear
            np.std(safe_contorl_cost),         # Safe-DDPG
            np.std(control_cost)               # RLC-FT
        ],
        'unit': 'MVar'
    },
    'Total Objective Cost': {
        'means': [
            np.mean(base_object_cost),         # Linear
            np.mean(safe_object_cost),         # Safe-DDPG
            np.mean(object_cost)               # RLC-FT
        ],
        'stds': [
            np.std(base_object_cost),          # Linear
            np.std(safe_object_cost),          # Safe-DDPG
            np.std(object_cost)                # RLC-FT
        ],
        'unit': ''
    }
}

# Print calculated results for verification (including truncation info)
print("Calculated Statistics:")
truncation_info = []
for metric in metrics:
    print(f"\n{metric}:")
    for i, method in enumerate(methods):
        mean_val = data[metric]['means'][i]
        std_val = data[metric]['stds'][i]
        lower_bound = mean_val - std_val
        
        if lower_bound < 0:
            truncation_info.append(f"{metric} - {method}")
            print(f"  {method}: {mean_val:.2f} ± {std_val:.2f} (truncated at 0)")
        else:
            print(f"  {method}: {mean_val:.2f} ± {std_val:.2f}")

# Nature-style colors
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

# Create subplots
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[metric for metric in metrics],  # Only metric names, no units
    horizontal_spacing=0.15
)

# Create grouped bar charts for each metric
for i, metric in enumerate(metrics):
    means = data[metric]['means']
    stds = data[metric]['stds']
    
    for j, method in enumerate(methods):
        mean_val = means[j]
        std_val = stds[j]
        
        # Check if truncation is needed (handle negative values)
        is_truncated = (mean_val - std_val) < 0
        
        # Calculate error bar lengths
        if is_truncated:
            # Truncation handling: lower error bar only extends to 0
            error_minus = mean_val  # distance from mean to 0
            error_plus = std_val    # keep original upper error bar
        else:
            # Normal case: symmetric error bars
            error_minus = std_val
            error_plus = std_val
        
        fig.add_trace(
            go.Bar(
                x=[method],
                y=[mean_val],
                error_y=dict(
                    type='data',
                    symmetric=False,
                    array=[error_plus],      # upper error bar
                    arrayminus=[error_minus], # lower error bar (may be truncated)
                    visible=True,
                    thickness=3,
                    width=5
                ),
                name=method,
                marker_color=colors[j],
                marker_line=dict(width=0.8, color='black'),
                showlegend=(i == 0),
                width=0.6
            ),
            row=1, col=i+1
        )
        
        # Add value labels above error bars
        if i == 0:  #
            fig.add_annotation(
                x=method,
                y=mean_val + max(means) * 0.05,  # above error bars
                xshift=25,  # shift value labels to the right (in pixels, adjustable)
                text=f"{mean_val:.1f}",
                showarrow=False,
                font=dict(size=14, color='black'),
                row=1, col=i+1
            )

        if i == 1:  #
            fig.add_annotation(
                x=method,
                y=mean_val + max(means) * 0.05,  # above error bars
                xshift=30,  # shift value labels to the right (in pixels, adjustable)
                text=f"{mean_val:.1f}",
                showarrow=False,
                font=dict(size=14, color='black'),
                row=1, col=i+1
            )

        if i == 2:  #
            fig.add_annotation(
                x=method,
                y=mean_val + max(means) * 0.05,  # above error bars
                xshift=40,  # shift value labels to the right (in pixels, adjustable)
                text=f"{mean_val:.1f}",
                showarrow=False,
                font=dict(size=14, color='black'),
                row=1, col=i+1
            )
        
        # Add truncation marker at bottom if truncated
        if is_truncated:
            fig.add_annotation(
                x=method,
                y=max(means) * 0.02,  # near bottom
                text="⊥",  # truncation symbol
                showarrow=False,
                font=dict(size=16, color='red'),
                row=1, col=i+1
            )

# Update layout (removed title)
fig.update_layout(
    width=1200,
    height=500,
    font=dict(
        family="Arial, sans-serif",
        size=16,
        color="black"
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        x=1.02,
        y=1,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255,255,255,0)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=14)
    ),
    margin=dict(l=70, r=140, t=80, b=70)  # reduced top margin since no title
)

# Update axis styles with units on y-axes
y_axis_titles = [
    data['Voltage Recovery Time']['unit'],
    data['Reactive Power']['unit'], 
    data['Total Objective Cost']['unit']
]

for i in range(1, 4):
    fig.update_xaxes(
        row=1, col=i,
        showgrid=False,
        showline=True,
        linewidth=1.5,
        linecolor='black',
        tickfont=dict(size=14)
    )
    
    # Set y-axis title with unit
    y_title = y_axis_titles[i-1] if y_axis_titles[i-1] else ""
    
    fig.update_yaxes(
        row=1, col=i,
        title=dict(
            text=y_title,
            font=dict(size=14, color='black')
        ),
        showgrid=True,
        gridwidth=0.5,
        gridcolor='lightgray',
        showline=True,
        linewidth=1.5,
        linecolor='black',
        tickfont=dict(size=14),
        rangemode='tozero'  # ensure y-axis starts from 0
    )

# Update subplot title formatting (without units now)
for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(size=16, color='black')

# Display figure
fig.show()

# Display truncation information
if truncation_info:
    print(f"\n⊥ Truncation Applied (error bars cut at zero for):")
    for info in truncation_info:
        print(f"  - {info}")
    print("Note: Red ⊥ symbols indicate where error bars were truncated at zero")

# Calculate performance improvement percentages
print("\n" + "="*50)
print("Performance Improvement of RLC-FT vs Linear:")
recovery_improve = (data['Voltage Recovery Time']['means'][0] - data['Voltage Recovery Time']['means'][2]) / data['Voltage Recovery Time']['means'][0] * 100
power_improve = (data['Reactive Power']['means'][0] - data['Reactive Power']['means'][2]) / data['Reactive Power']['means'][0] * 100
cost_improve = (data['Total Objective Cost']['means'][0] - data['Total Objective Cost']['means'][2]) / data['Total Objective Cost']['means'][0] * 100

print(f"• Voltage Recovery: {recovery_improve:.1f}% faster")
print(f"• Reactive Power: {power_improve:.1f}% reduction")
print(f"• Total Cost: {cost_improve:.1f}% reduction")

# Data integrity check
print("\n" + "="*50)
print("Data Quality Summary:")
for metric in metrics:
    means = data[metric]['means']
    stds = data[metric]['stds']
    
    print(f"\n{metric}:")
    for j, method in enumerate(methods):
        cv = (stds[j] / means[j]) * 100  # coefficient of variation
        truncated = "Yes" if (means[j] - stds[j]) < 0 else "No"
        print(f"  {method}: CV={cv:.1f}%, Truncated={truncated}")

# Save high-quality images (optional)
fig.write_image(os.path.join(Config.data_path, "images","56bus", "performance_comparison_123.png"), width=1200, height=500, scale=2)
fig.write_image(os.path.join(Config.data_path, "images","56bus", "performance_comparison_123.pdf"), width=1200, height=500)

Calculated Statistics:

Voltage Recovery Time:
  Linear: 106.38 ± 38.84
  Safe-DDPG: 51.59 ± 23.62
  RLC-FT: 17.99 ± 12.10

Reactive Power:
  Linear: 2034.35 ± 1696.93
  Safe-DDPG: 1087.41 ± 1017.19
  RLC-FT: 271.49 ± 287.73 (truncated at 0)

Total Objective Cost:
  Linear: 320150.54 ± 183398.67
  Safe-DDPG: 177025.43 ± 101214.86
  RLC-FT: 50326.50 ± 28921.66



⊥ Truncation Applied (error bars cut at zero for):
  - Reactive Power - RLC-FT
Note: Red ⊥ symbols indicate where error bars were truncated at zero

Performance Improvement of RLC-FT vs Linear:
• Voltage Recovery: 83.1% faster
• Reactive Power: 86.7% reduction
• Total Cost: 84.3% reduction

Data Quality Summary:

Voltage Recovery Time:
  Linear: CV=36.5%, Truncated=No
  Safe-DDPG: CV=45.8%, Truncated=No
  RLC-FT: CV=67.2%, Truncated=No

Reactive Power:
  Linear: CV=83.4%, Truncated=No
  Safe-DDPG: CV=93.5%, Truncated=No
  RLC-FT: CV=106.0%, Truncated=Yes

Total Objective Cost:
  Linear: CV=57.3%, Truncated=No
  Safe-DDPG: CV=57.2%, Truncated=No
  RLC-FT: CV=57.5%, Truncated=No
